In [1]:
import pandas as pd
import os
from langchain_community.graphs import Neo4jGraph
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from langchain_community.llms import Ollama
from dotenv import load_dotenv

In [ ]:
# llm = ChatOpenAI(temperature=0, model="gpt-4-0125-preview")

llm = Ollama(model="llama3", ) # Tested with llama3


# allowed_nodes = ["Law", "Section", "LegalConcept", "Penalty"]

# allowed_relationships = [
#     ("Section", "CITES", "Section"),        # Explicit cross-reference
#     ("Section", "REFERENCES", "Section"),   # General mention
#     ("Section", "DEFINES", "LegalConcept"), # What the section is about
#     ("Section", "PART_OF", "Law"),          # Structural hierarchy
#     ("Section", "HAS_PENALTY", "Penalty")   # Connection to punishment
# ]

llm_transformer = LLMGraphTransformer(llm=llm, ) # Model ต้องรองรับ tool เพื่อรองรับ Node properties

In [81]:
from neo4j import GraphDatabase

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE_NAME = "d6892fbf"


with GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD)) as driver:
    # 2. Verify connectivity to ensure a connection can be established
    driver.verify_connectivity()
    print("Connection established successfully.")
    
graph = Neo4jGraph(url=URI, username=USERNAME, password=PASSWORD,database=DATABASE_NAME)

Connection established successfully.


In [23]:
df = pd.read_parquet('../data/processed/')
df.head()

,law_name,section_content,section_num
0,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546,พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มา...,132
1,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5\nถ้าผู...,1598/5
2,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 876\nถ้าผู้รั...,876
3,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1030\nถ้าผู้เ...,1030
4,ประมวลกฎหมายแพ่งและพาณิชย์,ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 849\nการรับเง...,849


In [49]:
len(df)

3833

In [28]:
df.iloc[0,:]

law_name                 พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546
section_content    พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มา...
section_num                                                      132
Name: 0, dtype: str

In [ ]:
text = ""
col = df.columns.to_list()
for i in range(10):
    sub_text = ""
    for j in range(len(col)-1):
        sub_text += df.iloc[i, j]
    text += sub_text
    text += "\n"
print(text)
# for t in range(len(text)):
#     print(text[t])
#     print()

พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546 มาตรา 132 ผู้ใดประกอบกิจการในลักษณะเป็นศูนย์ซื้อขายสัญญาซื้อขายล่วงหน้าโดยไม่ได้รับใบอนุญาตหรือไม่ได้จดทะเบียนตามมาตรา 54 ต้องระวางโทษจำคุกไม่เกินสามปี หรือปรับไม่เกินสามแสนบาท หรือทั้งจำทั้งปรับ และปรับอีกไม่เกินวันละหนึ่งหมื่นบาทตลอดเวลาที่ยังฝ่าฝืน
ประมวลกฎหมายแพ่งและพาณิชย์ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 1598/5
ถ้าผู้อยู่ในปกครองรู้จักผิดชอบและมีอายุไม่ต่ำกว่าสิบห้าปีบริบูรณ์เมื่อผู้ปกครองจะทำกิจการใดที่สำคัญ ให้ปรึกษาหารือผู้อยู่ในปกครองก่อนเท่าที่จะทำได้
การที่ผู้อยู่ในปกครองได้ยินยอมด้วยนั้นหาคุ้มผู้ปกครองให้พ้นจากความรับผิดไม่
ประมวลกฎหมายแพ่งและพาณิชย์ประมวลกฎหมายแพ่งและพาณิชย์ มาตรา 876
ถ้าผู้รับประกันภัยต้องคำพิพากษาให้เป็นคนล้มละลาย ผู้เอาประกันภัยจะเรียกให้หาประกันอันสมควรให้แก่ตนก็ได้ หรือจะบอกเลิกสัญญาเสียก็ได้
ถ้าผู้เอาประกันภัยต้องคำพิพากษาให้เป็นคนล้มละลาย ท่านให้ใช้วิธีเดียวกันนี้บังคับตามควรแก่เรื่อง แต่กระนั้นก็ดี ถ้าเบี้ยประกันภัยได้ส่งแล้วเต็มจำนวนเพื่ออายุประกันภัยเป็นระยะเวล

In [ ]:
documents = [Document(page_content=text)]
graph_documents = llm_transformer.convert_to_graph_documents(documents)

# 46 sec for 10 rows of data with llama3 to convert text to graph

In [54]:
for node in graph_documents[0].nodes:
    print(node)

id='มาตรา 132' type='Section' properties={}
id='มาตรา 21' type='Section' properties={}
id='มาตรา 565' type='Section' properties={}
id='มาตรา 1030' type='Section' properties={}
id='มาตรา 1598/26' type='Section' properties={}
id='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546' type='Law' properties={}
id='มาตรา 54' type='Section' properties={}
id='มาตรา 876' type='Section' properties={}
id='ประมวลกฎหมายแพ่งและพาณิชย์' type='Law' properties={}
id='พระราชบัญญัติหลักประกันทางธุรกิจ พ.ศ. 2558' type='Law' properties={}
id='มาตรา 9' type='Section' properties={}
id='มาตรา 849' type='Section' properties={}
id='มาตรา 193' type='Section' properties={}
id='พระราชบัญญัติการบัญชี พ.ศ. 2543' type='Law' properties={}
id='มาตรา 1598/5' type='Section' properties={}


In [55]:
for relationship in graph_documents[0].relationships:
    print(relationship)

source=Node(id='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', type='Law', properties={}) target=Node(id='มาตรา 132', type='Section', properties={}) type='DEFINED_BY' properties={}
source=Node(id='พระราชบัญญัติสัญญาซื้อขายล่วงหน้า พ.ศ. 2546', type='Law', properties={}) target=Node(id='มาตรา 54', type='Section', properties={}) type='DEFINED_BY' properties={}
source=Node(id='ประมวลกฎหมายแพ่งและพาณิชย์', type='Law', properties={}) target=Node(id='มาตรา 1598/5', type='Section', properties={}) type='CONTAINS' properties={}
source=Node(id='ประมวลกฎหมายแพ่งและพาณิชย์', type='Law', properties={}) target=Node(id='มาตรา 876', type='Section', properties={}) type='CONTAINS' properties={}
source=Node(id='ประมวลกฎหมายแพ่งและพาณิชย์', type='Law', properties={}) target=Node(id='มาตรา 1030', type='Section', properties={}) type='CONTAINS' properties={}
source=Node(id='ประมวลกฎหมายแพ่งและพาณิชย์', type='Law', properties={}) target=Node(id='มาตรา 849', type='Section', properties={}) type='CONTAINS' propert

In [61]:
graph.add_graph_documents(graph_documents)